# SecureNeuroGrade — Brain Tumor MRI Model Training (CAFNet)
**CAFNet: Hybrid CNN–ViT Framework with Cross-Attention Fusion for Robust Brain Tumor Classification**  
Based on *Jayaraman et al., Scientific Reports 15:45769 (2025)* and SecureNeuroGrade `MODEL.md` specification.

**Run on:** Google Colab (Runtime → Change runtime type → **T4 GPU**) or Kaggle (Accelerator → GPU).  
Upload `archive__8__3.zip` or `archive (8).zip` (Colab: left sidebar → Files, or Google Drive; Kaggle: add as a dataset).

**Target Classes (must strictly match `config.json`):**
- `0`: **glioma**
- `1`: **meningioma**
- `2`: **pituitary**
- `3`: **no_tumor**

> ⚠️ **Medical AI Boundary (AGENTS.md §5):** SecureNeuroGrade is an academic/clinical decision-support prototype, NOT an autonomous medical diagnosis system. Model predictions assist qualified radiologists and must never replace clinical judgement.

In [ ]:
!pip install -q tensorflow scikit-learn matplotlib seaborn pillow opencv-python
import tensorflow as tf
print("TensorFlow Version:", tf.__version__)
print("Available GPUs:    ", tf.config.list_physical_devices('GPU'))
if not tf.config.list_physical_devices('GPU'):
    print("⚠️ WARNING: No GPU detected. In Colab, go to Runtime -> Change runtime type -> T4 GPU.")
else:
    print("✅ GPU ready for CAFNet training.")

## 1. Locate & Unzip the Dataset
Supports Google Drive mounting, local file upload, or automated KaggleHub download.

In [ ]:
# Mount Google Drive so the notebook can find your dataset zip (skip if you use Kaggle or the kagglehub cell below).
# Put "archive (8).zip" or "archive__8__3.zip" anywhere under My Drive.
try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    print("Not on Colab - skipping Drive mount.")

In [ ]:
# OPTIONAL: skip uploading entirely and pull the dataset straight into Colab with kagglehub.
# Needs a Kaggle account (Colab secrets KAGGLE_USERNAME / KAGGLE_KEY, or kagglehub's login prompt).
# Uncomment to use:
#
# !pip install -q kagglehub
# import kagglehub
# path = kagglehub.dataset_download("masoudnickparvar/brain-tumor-mri-dataset")
# print("Downloaded to:", path)
# import shutil; shutil.copytree(path, "dataset", dirs_exist_ok=True)

In [ ]:
import os, glob, zipfile

ZIP_NAMES = ["archive (8).zip", "archive__8__3.zip"]
WORK_DIR = "dataset"

def valid_zip(p):
    """True only if the file is a complete, readable zip (catches truncated uploads)."""
    try:
        if os.path.getsize(p) == 0:
            return False
        with zipfile.ZipFile(p) as z:
            return z.testzip() is None
    except (zipfile.BadZipFile, EOFError):
        return False

candidates = []
for zip_name in ZIP_NAMES:
    candidates.extend(glob.glob(f"/content/{zip_name}") +
                      glob.glob(f"/kaggle/input/**/{zip_name}", recursive=True) +
                      glob.glob(f"./{zip_name}") +
                      glob.glob(f"/content/drive/MyDrive/**/{zip_name}", recursive=True))
candidates = list(dict.fromkeys(candidates))   # de-duplicate, keep order

good = [f for f in candidates if valid_zip(f)]
for f in candidates:
    if f not in good:
        print(f"Skipping corrupt/incomplete zip: {f} ({os.path.getsize(f)/1e6:.1f} MB) "
              f"- upload was probably cut off, re-upload it (Google Drive is more reliable).")

if os.path.isdir(os.path.join(WORK_DIR, "Training")):
    print("Dataset folder already extracted and present.")
elif good:
    ZIP_PATH = good[0]
    print(f"Extracting {ZIP_PATH} ({os.path.getsize(ZIP_PATH)/1e6:.1f} MB)...")
    with zipfile.ZipFile(ZIP_PATH) as z:
        z.extractall(WORK_DIR)
else:
    raise FileNotFoundError(
        "No valid zip found. Upload a complete archive (Drive recommended: "
        "from google.colab import drive; drive.mount('/content/drive')), "
        "or run the optional Kaggle download cell above, then re-run this cell.")

# Some zips nest everything one level deeper - handle that
if not os.path.isdir(os.path.join(WORK_DIR, "Training")):
    for root, dirs, _ in os.walk(WORK_DIR):
        if "Training" in dirs and "Testing" in dirs:
            WORK_DIR = root
            break

print("\n--- Dataset Distribution ---")
for split in ("Training", "Testing"):
    for c in sorted(os.listdir(os.path.join(WORK_DIR, split))):
        print(f"{split:8s} | {c:12s} | {len(os.listdir(os.path.join(WORK_DIR, split, c))):4d} scans")

In [ ]:
# Sanity check: confirm 4 training classes
print('Training classes found:', sorted(os.listdir(os.path.join(WORK_DIR, 'Training'))))

## 2. Configuration & CAFNet Hyperparameters
Configures training constants and CAFNet architectural parameters:
- **CNN Backbone**: EfficientNetB0 (ImageNet pretrained, frozen in Phase 1)
- **ViT Branch**: 16x16 patch projection (196 tokens), 128-dim embeddings, 2 Transformer Encoder layers with 4-head self-attention
- **Cross-Attention Fusion (CAF)**: 4-head cross-attention where Query = ViT (global context) and Key/Value = CNN (local textures)
- **Classifier Head**: Concatenation of pooled fused cross-attention tokens and pooled CNN features -> Dense -> Softmax (4 classes)

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt

# Image and Batching
IMAGE_SIZE   = (224, 224)
BATCH_SIZE   = 32
NUM_CLASSES  = 4
SEED         = 42

# Training Schedule
EPOCHS_HEAD  = 10      # Phase 1: Train ViT branch, CAF module & classifier head
EPOCHS_FT    = 15      # Phase 2: End-to-end fine-tuning with top CNN layers unfrozen
LR_HEAD      = 1e-4    # Learning rate for Phase 1
LR_FT        = 1e-5    # Lower learning rate for Phase 2 fine-tuning

# CAFNet Hyperparameters (following Jayaraman et al. 2025 / cafnet_seminar.pptx)
PATCH_SIZE         = 16    # 16x16 patches -> 14x14 = 196 patches for 224x224 input
PROJECTION_DIM     = 128   # Common embedding dimension for ViT & CNN projected tokens
NUM_HEADS          = 4     # Number of attention heads (Self-Attention & Cross-Attention)
TRANSFORMER_LAYERS = 2     # Number of ViT Transformer Encoder blocks
MLP_DIM            = 256   # Hidden dimension in Transformer & CAF Feed-Forward Networks

# Folder names on disk -> index order. 'notumor' (index 3) is reported as 'no_tumor'.
FOLDER_CLASSES = ['glioma', 'meningioma', 'pituitary', 'notumor']
CLASSES        = ['glioma', 'meningioma', 'pituitary', 'no_tumor']

OUT_DIR = "ml/models/brain_tumor/checkpoints"
MODEL_PATH = os.path.join(OUT_DIR, "brain_tumor_model.keras")
os.makedirs(OUT_DIR, exist_ok=True)
tf.keras.utils.set_random_seed(SEED)
print(f"CAFNet configuration initialized. Checkpoint destination: {MODEL_PATH}")

## 3. Data Loading & CAFNet Data Augmentation
Following **cafnet_seminar.pptx (Slide 8)** and *Jayaraman et al. (2025)*:
- **Rescaling**: `[0.0, 1.0]` (pixel values / 255.0) matching `inference_service.py`
- **Random Rotation**: $\pm 20^\circ$ (simulates real-world scan orientation variance)
- **Random Zoom**: $\pm 20\%$ (simulates field-of-view differences)
- **Horizontal Flip**: 50% probability
- **Evaluation Split**: 85/15 stratified train/validation on `Training/`; held-out `Testing/` evaluated without augmentation.

In [ ]:
def load(subdir, **kw):
    return tf.keras.utils.image_dataset_from_directory(
        os.path.join(WORK_DIR, subdir), labels="inferred", label_mode="categorical",
        class_names=FOLDER_CLASSES, color_mode="rgb", image_size=IMAGE_SIZE,
        batch_size=BATCH_SIZE, seed=SEED, **kw)

train_raw = load("Training", validation_split=0.15, subset="training", shuffle=True)
val_raw   = load("Training", validation_split=0.15, subset="validation", shuffle=False)
test_raw  = load("Testing", shuffle=False)

# CAFNet Augmentation pipeline strictly matching cafnet_seminar.pptx Slide 8
augment = tf.keras.Sequential([
    tf.keras.layers.RandomRotation(0.055, name="rot_20deg"),   # 0.055 * 360° ≈ 20°
    tf.keras.layers.RandomZoom(0.2, name="zoom_20pct"),        # ±20% zoom
    tf.keras.layers.RandomFlip("horizontal", name="h_flip_50pct"), # 50% horizontal flip
], name="cafnet_augmentation")

AUTOTUNE = tf.data.AUTOTUNE
norm = lambda x, y: (tf.cast(x, tf.float32) / 255.0, y)

train_ds = (train_raw.map(lambda x, y: (augment(x, training=True), y), num_parallel_calls=AUTOTUNE)
                     .map(norm, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE))
val_ds   = val_raw.map(norm, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)
test_ds  = test_raw.map(norm, num_parallel_calls=AUTOTUNE).prefetch(AUTOTUNE)

# Visual sanity check: display 6 augmented training samples
imgs, lbls = next(iter(train_raw))
plt.figure(figsize=(14, 3))
for i in range(6):
    plt.subplot(1, 6, i+1)
    plt.imshow(imgs[i].numpy().astype("uint8"))
    plt.title(CLASSES[int(tf.argmax(lbls[i]))], fontsize=10)
    plt.axis("off")
plt.suptitle("Augmented Training Samples (CAFNet Pipeline)", y=1.05, fontsize=12)
plt.show()

## 4. Build the CAFNet Model (Hybrid CNN–ViT with Cross-Attention Fusion)

### Architectural Rationale (*cafnet_seminar.pptx* Slides 7, 9 & 10):
1. **Local vs. Global Dilemma**:
   - **CNNs** excel at capturing fine-grained local textures and tumor margins, but struggle with long-range anatomical dependencies across the entire brain.
   - **Vision Transformers (ViT)** model global contextual relationships through self-attention, but lack inductive spatial biases and suffer on moderate-sized medical datasets.
2. **Cross-Attention Fusion (CAF)**:
   - ViT global context acts as the **Query ($Q$)**: *"Which local regions matter for the overall diagnosis?"*
   - CNN local feature maps act as the **Key ($K$) and Value ($V$)**: providing the fine-grained spatial textures queried by ViT.
   - Fuses local texture and global context dynamically before final classification.
3. **Inference Consistency**:
   - Accepts normalized `[0.0, 1.0]` inputs matching `MODEL.md` §2.2 and `app/services/inference_service.py`.
   - Produces the exact 4-class Softmax probability distribution: `[P(glioma), P(meningioma), P(pituitary), P(no_tumor)]`.

In [ ]:
from tensorflow.keras import layers, models
from tensorflow.keras.applications import EfficientNetB0

@tf.keras.utils.register_keras_serializable(package="CAFNet")
class PositionEmbedding(layers.Layer):
    """Learnable 1D positional embedding for patch tokens."""
    def __init__(self, num_patches, projection_dim, **kwargs):
        super().__init__(**kwargs)
        self.num_patches = num_patches
        self.projection_dim = projection_dim

    def build(self, input_shape):
        self.pos_emb = self.add_weight(
            name="pos_emb",
            shape=(1, self.num_patches, self.projection_dim),
            initializer="zeros",
            trainable=True,
        )
        super().build(input_shape)

    def call(self, x):
        return x + self.pos_emb

    def get_config(self):
        config = super().get_config()
        config.update({
            "num_patches": self.num_patches,
            "projection_dim": self.projection_dim,
        })
        return config


def build_cafnet(
    input_shape=(224, 224, 3),
    num_classes=4,
    patch_size=16,
    projection_dim=128,
    num_heads=4,
    transformer_layers=2,
    mlp_dim=256
):
    """
    Constructs CAFNet: Hybrid CNN-ViT Framework with Cross-Attention Fusion.
    Reference: Jayaraman et al., Scientific Reports 15:45769 (2025) / cafnet_seminar.pptx
    """
    inputs = layers.Input(shape=input_shape, name="mri_input")  # [0.0, 1.0] float32

    # =========================================================================
    # Branch 1: CNN Local Feature Extractor (Keys and Values)
    # =========================================================================
    cnn_in = layers.Rescaling(255.0, name="cnn_rescaling")(inputs)  # EfficientNet expects [0, 255]
    base_cnn = EfficientNetB0(weights="imagenet", include_top=False, input_shape=input_shape)
    base_cnn.trainable = False
    cnn_feat = base_cnn(cnn_in, training=False)  # Shape: (B, 7, 7, 1280)

    # 1x1 Conv to project CNN channels to projection_dim
    cnn_proj = layers.Conv2D(projection_dim, kernel_size=1, padding="same", name="cnn_proj_1x1")(cnn_feat)
    cnn_tokens = layers.Reshape((49, projection_dim), name="cnn_spatial_tokens")(cnn_proj)  # 7x7=49
    cnn_tokens = layers.LayerNormalization(epsilon=1e-6, name="cnn_token_norm")(cnn_tokens)

    # =========================================================================
    # Branch 2: Vision Transformer (ViT) Global Context Extractor (Queries)
    # =========================================================================
    num_patches = (input_shape[0] // patch_size) * (input_shape[1] // patch_size)  # 14x14 = 196
    vit_patches = layers.Conv2D(
        filters=projection_dim,
        kernel_size=patch_size,
        strides=patch_size,
        padding="valid",
        name="vit_patch_proj"
    )(inputs)  # Shape: (B, 14, 14, projection_dim)
    vit_tokens = layers.Reshape((num_patches, projection_dim), name="vit_tokens_flat")(vit_patches)
    vit_x = PositionEmbedding(num_patches=num_patches, projection_dim=projection_dim, name="vit_pos_embed")(vit_tokens)

    # Transformer Encoder Blocks (Multi-Head Self-Attention + MLP)
    for i in range(transformer_layers):
        norm_sa = layers.LayerNormalization(epsilon=1e-6, name=f"vit_norm_sa_{i+1}")(vit_x)
        sa = layers.MultiHeadAttention(
            num_heads=num_heads,
            key_dim=projection_dim // num_heads,
            name=f"vit_self_attn_{i+1}"
        )(norm_sa, norm_sa)
        vit_x = layers.Add(name=f"vit_res_sa_{i+1}")([sa, vit_x])

        norm_ffn = layers.LayerNormalization(epsilon=1e-6, name=f"vit_norm_ffn_{i+1}")(vit_x)
        ffn = layers.Dense(mlp_dim, activation="gelu", name=f"vit_ffn1_{i+1}")(norm_ffn)
        ffn = layers.Dropout(0.1, name=f"vit_drop_{i+1}")(ffn)
        ffn = layers.Dense(projection_dim, name=f"vit_ffn2_{i+1}")(ffn)
        vit_x = layers.Add(name=f"vit_res_ffn_{i+1}")([ffn, vit_x])

    vit_global = layers.LayerNormalization(epsilon=1e-6, name="vit_global_norm")(vit_x)  # (B, 196, projection_dim)

    # =========================================================================
    # Cross-Attention Fusion (CAF) Module
    # Q = ViT (global context), K & V = CNN (local textures)
    # =========================================================================
    caf_cross_attn = layers.MultiHeadAttention(
        num_heads=num_heads,
        key_dim=projection_dim // num_heads,
        name="caf_cross_attention"
    )(query=vit_global, key=cnn_tokens, value=cnn_tokens)

    caf_x = layers.Add(name="caf_res_cross")([caf_cross_attn, vit_global])
    caf_x = layers.LayerNormalization(epsilon=1e-6, name="caf_norm_cross")(caf_x)

    caf_ffn = layers.Dense(mlp_dim, activation="gelu", name="caf_ffn_dense1")(caf_x)
    caf_ffn = layers.Dropout(0.2, name="caf_ffn_drop")(caf_ffn)
    caf_ffn = layers.Dense(projection_dim, name="caf_ffn_dense2")(caf_ffn)
    caf_fused = layers.Add(name="caf_res_ffn")([caf_ffn, caf_x])
    caf_fused = layers.LayerNormalization(epsilon=1e-6, name="caf_norm_fused")(caf_fused)

    # =========================================================================
    # Classification Head (Fused CAF Context + Direct CNN Features)
    # =========================================================================
    fused_pool = layers.GlobalAveragePooling1D(name="caf_gap_1d")(caf_fused)  # (B, projection_dim)
    cnn_pool = layers.GlobalAveragePooling2D(name="cnn_gap_2d")(cnn_feat)
    cnn_pool_proj = layers.Dense(projection_dim, activation="relu", name="cnn_pool_proj")(cnn_pool)

    combined = layers.Concatenate(name="fusion_concat")([fused_pool, cnn_pool_proj])

    x = layers.Dense(128, activation="relu", name="head_dense")(combined)
    x = layers.BatchNormalization(name="head_bn")(x)
    x = layers.Dropout(0.3, name="head_dropout")(x)
    outputs = layers.Dense(num_classes, activation="softmax", name="probs")(x)

    model = models.Model(inputs=inputs, outputs=outputs, name="SecureNeuroGrade_BrainTumor_CAFNet")
    return model, base_cnn

model, base_cnn = build_cafnet(
    input_shape=(IMAGE_SIZE[0], IMAGE_SIZE[1], 3),
    num_classes=NUM_CLASSES,
    patch_size=PATCH_SIZE,
    projection_dim=PROJECTION_DIM,
    num_heads=NUM_HEADS,
    transformer_layers=TRANSFORMER_LAYERS,
    mlp_dim=MLP_DIM
)

def compile_model(lr):
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=lr),
        loss="categorical_crossentropy",
        metrics=[
            "accuracy",
            tf.keras.metrics.Precision(name="precision"),
            tf.keras.metrics.Recall(name="recall")
        ]
    )

compile_model(LR_HEAD)
model.summary()

## 5. Phase 1 — Train ViT Branch, Cross-Attention Fusion (CAF) & Classifier Head
In this phase, the CNN backbone (EfficientNetB0) is **frozen**.
Training focuses on establishing the patch embeddings, transformer self-attention blocks, the cross-attention fusion alignments, and the dense classification head at `LR = 1e-4`.

In [ ]:
def make_callbacks():
    return [
        tf.keras.callbacks.EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True, verbose=1),
        tf.keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, min_lr=1e-7, verbose=1),
        tf.keras.callbacks.ModelCheckpoint(MODEL_PATH, monitor="val_accuracy", save_best_only=True, verbose=1),
    ]

print("Starting Phase 1 training (CNN backbone frozen, optimizing ViT + CAF + Classifier)...\n")
hist1 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_HEAD, callbacks=make_callbacks())

## 6. Phase 2 — End-to-End Fine-Tuning (Unfreeze Top CNN Layers)
Unfreezes the top 40 layers of the CNN backbone (keeping `BatchNormalization` layers frozen for variance stability).
The full hybrid pipeline is trained end-to-end at a gentle learning rate (`LR = 1e-5`).

In [ ]:
print("Unfreezing top CNN layers for end-to-end fine-tuning...\n")
base_cnn.trainable = True
for layer in base_cnn.layers[:-40]:
    layer.trainable = False
for layer in base_cnn.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False

compile_model(LR_FT)   # Recompile model with fine-tune learning rate
hist2 = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS_FT, callbacks=make_callbacks())

## 7. Training Curves & Convergence Diagnostics
Visualizes accuracy and loss progression across Phase 1 and Phase 2.

In [ ]:
def cat(k): return hist1.history[k] + hist2.history[k]
fig, ax = plt.subplots(1, 2, figsize=(14, 4))
for a, k, t in zip(ax, ["accuracy", "loss"], ["Accuracy", "Loss"]):
    a.plot(cat(k), label="Train " + t, lw=2)
    a.plot(cat("val_" + k), label="Val " + t, lw=2)
    a.axvline(len(hist1.history[k]) - 1, ls="--", c="gray", label="Phase 2 Fine-Tune Starts")
    a.set_title(f"CAFNet {t} Progression", fontsize=12)
    a.set_xlabel("Epoch")
    a.grid(True, alpha=0.3)
    a.legend()
plt.tight_layout()
plt.show()

## 8. Final Evaluation on Held-out Testing Set
Evaluates the best checkpoint (selected via `val_accuracy`) on the strictly held-out `Testing/` directory.
Calculates test accuracy, multi-class classification report, and confusion matrix.

In [ ]:
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

best = tf.keras.models.load_model(MODEL_PATH)  # Best checkpoint by val_accuracy
loss, acc, prec, rec = best.evaluate(test_ds, verbose=0)
print(f"CAFNet Test Accuracy:  {acc*100:.2f}%")
print(f"CAFNet Test Loss:      {loss:.4f}")
print(f"CAFNet Test Precision: {prec*100:.2f}%")
print(f"CAFNet Test Recall:    {rec*100:.2f}%")

y_true = np.concatenate([np.argmax(y.numpy(), 1) for _, y in test_ds])
y_prob = best.predict(test_ds, verbose=0)
y_pred = y_prob.argmax(1)

print("\n" + "="*60)
print("Classification Report (Held-out Test Set):")
print("="*60)
print(classification_report(y_true, y_pred, target_names=CLASSES, digits=4))

cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", xticklabels=CLASSES, yticklabels=CLASSES)
plt.title("CAFNet Test Confusion Matrix", fontsize=12)
plt.xlabel("Predicted Class"); plt.ylabel("True Class")
plt.tight_layout()
plt.show()

## 9. Export `config.json` (Model Registry Metadata)
Saves the official model metadata file consumed by SecureNeuroGrade (`app/models/ml_models.py` and `app/services/inference_service.py`).

In [ ]:
from datetime import date

config = {
    "disease_type": "brain_tumor",
    "organ_type": "brain",
    "model_name": "brain-tumor-cafnet",
    "model_version": "v2.0",
    "description": "CAFNet (Hybrid CNN-ViT with Cross-Attention Fusion) for 4-class brain tumor MRI classification.",
    "architecture": {
        "framework": "CAFNet",
        "cnn_backbone": "EfficientNetB0",
        "vit_patches": 196,
        "patch_size": [16, 16],
        "projection_dim": PROJECTION_DIM,
        "cross_attention_heads": NUM_HEADS,
        "fusion_mechanism": "Cross-Attention (Query=ViT, Key=CNN, Value=CNN)",
        "reference": "Jayaraman et al., Scientific Reports 15:45769 (2025)"
    },
    "input_shape": [224, 224, 3],
    "classes": CLASSES,
    "preprocessing": {
        "resize": [224, 224],
        "normalize": True,
        "normalize_range": [0.0, 1.0],
        "color_mode": "rgb"
    },
    "training": {
        "dataset_version": "Kaggle-Brain-Tumor-MRI (uploaded archive)",
        "training_date": str(date.today()),
        "epochs": len(hist1.history["loss"]) + len(hist2.history["loss"]),
        "batch_size": BATCH_SIZE,
        "optimizer": "adam",
        "learning_rate": LR_HEAD,
        "fine_tune_learning_rate": LR_FT,
        "test_accuracy": round(float(acc), 4),
        "test_loss": round(float(loss), 4),
        "test_precision": round(float(prec), 4),
        "test_recall": round(float(rec), 4)
    }
}

os.makedirs("ml/models/brain_tumor", exist_ok=True)
with open("ml/models/brain_tumor/config.json", "w") as f:
    json.dump(config, f, indent=4)
print("Saved ml/models/brain_tumor/config.json:")
print(json.dumps(config, indent=2))

## 10. Integration Smoke Test
Validates end-to-end inference using the exact preprocessing pipeline from `MODEL.md` §2.3 and verifies that the output matches the dictionary structure required by Module M3 (`classification_service.py`).

In [ ]:
import cv2

def preprocess_mri_scan(path, target_size=(224, 224)):
    img = cv2.imread(path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, target_size, interpolation=cv2.INTER_AREA)
    return np.expand_dims(img.astype(np.float32) / 255.0, 0)

def run_inference(path, model=best):
    probs = model.predict(preprocess_mri_scan(path), verbose=0)[0]
    preds = {c: float(p) for c, p in zip(CLASSES, probs)}
    top = max(preds, key=preds.get)
    return {
        "predictions": preds,
        "predicted_class": top,
        "confidence": preds[top],
        "is_placeholder": False,
        "organ_type": "brain",
        "model_architecture": "CAFNet"
    }

print("Running integration smoke test across sample test scans for each class:")
for cls in FOLDER_CLASSES:
    sample_files = glob.glob(os.path.join(WORK_DIR, "Testing", cls, "*"))
    if sample_files:
        f = sorted(sample_files)[0]
        r = run_inference(f)
        print(f"True: {cls:11s} -> Predicted: {r['predicted_class']:11s} (Confidence: {r['confidence']:.4f})")

## 11. Download the Artifacts
Downloads the trained `.keras` model checkpoint and `config.json` for placement into `ml/models/brain_tumor/checkpoints/` in SecureNeuroGrade.

In [ ]:
# Colab: download directly. Kaggle: files appear under 'Output' in /kaggle/working.
try:
    from google.colab import files
    files.download(MODEL_PATH)
    files.download("ml/models/brain_tumor/config.json")
except ImportError:
    print("Not on Colab — grab", MODEL_PATH, "and ml/models/brain_tumor/config.json from the file browser.")

## 12. Architectural Summary & Benchmark Comparison

### Performance Comparison (*cafnet_seminar.pptx* Slide 16)
| Architecture | Test Accuracy | Strengths | Limitations |
| :--- | :---: | :--- | :--- |
| **Classical ML (SVM / Random Forest)** | 82.99% | Simple, fast training | Relies on manual, hand-crafted features |
| **Plain CNN (Vanilla ConvNet)** | 75.90% | Strong local edge detection | Misses global anatomical context |
| **Standalone ViT** | 87.34% | Long-range context awareness | Weak inductive bias; prone to overfitting on small data |
| **CAFNet (Hybrid CNN–ViT with CAF)** | **96.41%** | **Dual local+global representations with query-driven cross-attention** | Higher training FLOPs |

### Core Findings:
1. **Ablation Insight**: The ablation experiments confirm that the **Cross-Attention Fusion (CAF) mechanism itself**—rather than merely concatenating two models—drives the majority of the +9.07% accuracy boost over standalone ViT and +20.51% over plain CNN.
2. **Clinical Decision Support (AGENTS.md §5)**: CAFNet serves as an objective, high-precision decision-support tool for authorized radiologists in SecureNeuroGrade. Final diagnosis remains the exclusive domain of licensed medical personnel.